# AI Functions in Production: Evaluate → Optimize → Monitor

**Use case:** support-ticket triage with `AI_COMPLETE` structured output. One call returns
`product`, `issue_type`, `sentiment`, `requested_action`, `order_id`, and an `escalation_flag`.

| Phase | What we do | Snowflake capability |
|---|---|---|
| 1. Build | Wrap the prompt as a governed, reusable function | `CREATE AI FUNCTION` + `AI_COMPLETE(response_format=>...)` |
| 2. Evaluate | Score against a labeled golden set *before* production | `CREATE EXPERIMENT ... TYPE='AI_FUNCTION_EVALUATION'` |
| 3. Optimize | Auto-tune the prompt and compare models on cost vs quality | `CREATE EXPERIMENT ... TYPE='AI_FUNCTION_OPTIMIZATION'` |
| 4. Deploy | Promote the winner straight from the experiment | `CREATE AI FUNCTION ... FROM EXPERIMENT` |
| 5. Monitor | Cost, output health, drift, sampled quality, alerts | `CORTEX_AI_FUNCTIONS_USAGE_HISTORY`, tasks, alerts |

Prereq: run `setup.sql` (creates `AI_FUNCTIONS_EVAL_DEMO` and synthetic labeled tickets).

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.use_schema("AI_FUNCTIONS_EVAL_DEMO.EVAL")

# Set RUN_LIVE = True to launch new experiments (optimization takes ~10 min).
# With False, the notebook reads the pre-run experiments below.
RUN_LIVE = False
SUFFIX   = "LIVE" if RUN_LIVE else ""
EVAL_EXP = "TRIAGE_BASELINE_EVAL" + ("_" + SUFFIX if SUFFIX else "")
OPT_EXP  = "TRIAGE_TICKET_OPT_EXP" + ("_" + SUFFIX if SUFFIX else "")

## 1. The data: a labeled golden set
Every row has the ticket text and the **expected** structured output. Ground truth encodes the business rule:
*escalate when it's a safety issue, a legal threat, or a negative customer contacting us again.*

In [ ]:
%%sql -r golden_set
SELECT ticket_text, expected FROM AI_FUNCTIONS_EVAL_DEMO.EVAL.HOLDOUT LIMIT 5

## 2. Baseline: a naive prompt with structured output
`response_format` guarantees **valid JSON with the right keys**, but not that the *values* are right.
The baseline prompt doesn't state the allowed categories or the escalation rule.

In [ ]:
%%sql -r baseline_ddl
CREATE OR REPLACE AI FUNCTION AI_FUNCTIONS_EVAL_DEMO.EVAL.TRIAGE_TICKET(TICKET_TEXT VARCHAR)
RETURNS VARIANT
COMMENT = 'Extract structured fields, sentiment, and an escalation flag from a support ticket'
AS
$$
  SELECT AI_COMPLETE(
    model => 'claude-sonnet-4-6',
    messages => ARRAY_CONSTRUCT(
      OBJECT_CONSTRUCT('role','system','content','You are a customer support assistant. Read the ticket and fill in the requested fields.'),
      OBJECT_CONSTRUCT('role','user','content','Ticket: ' || TICKET_TEXT)),
    response_format => PARSE_JSON('{"type":"json","schema":{"type":"object","properties":{
        "product":{"type":"string","description":"product name"},
        "issue_type":{"type":"string","description":"type of issue"},
        "sentiment":{"type":"string","description":"customer sentiment"},
        "requested_action":{"type":"string","description":"what the customer wants"},
        "order_id":{"type":"string","description":"order number if present"},
        "escalation_flag":{"type":"boolean","description":"should this be escalated"}},
      "required":["product","issue_type","sentiment","requested_action","order_id","escalation_flag"],
      "additionalProperties":false}}'))
$$

In [ ]:
%%sql -r baseline_preview
SELECT ticket_text, AI_FUNCTIONS_EVAL_DEMO.EVAL.TRIAGE_TICKET(ticket_text) AS predicted, expected
FROM AI_FUNCTIONS_EVAL_DEMO.EVAL.HOLDOUT LIMIT 3

Eyeballing three rows looks "pretty good": the JSON is valid and the product is right. But `issue_type` comes back as
*"Setup / Feature Configuration"* instead of `how_to`, and sentiment as *"Frustrated"*. Downstream routing and dashboards
break on values like these. **That's why we measure instead of eyeballing.**

## 3. Evaluate before production
1. Freeze the golden set as a versioned, immutable **DATASET** so every score is reproducible.
2. Run an **evaluation experiment** with an LLM judge comparing predicted vs expected.

In [ ]:
if RUN_LIVE:
    session.sql("CREATE DATASET IF NOT EXISTS AI_FUNCTIONS_EVAL_DEMO.EVAL.TICKETS_HOLDOUT").collect()
    session.sql("CREATE DATASET IF NOT EXISTS AI_FUNCTIONS_EVAL_DEMO.EVAL.TICKETS_TRAIN").collect()
    # versions are immutable; add a new one whenever the golden set changes
    for ds, tbl in [("TICKETS_HOLDOUT", "HOLDOUT"), ("TICKETS_TRAIN", "TRAIN")]:
        try:
            session.sql(f"ALTER DATASET AI_FUNCTIONS_EVAL_DEMO.EVAL.{ds} ADD VERSION 'v1' FROM (SELECT * FROM AI_FUNCTIONS_EVAL_DEMO.EVAL.{tbl})").collect()
        except Exception as e:
            print(f"{ds} v1 already exists, reusing it")
    session.sql(f"""
CREATE EXPERIMENT {EVAL_EXP}
  TYPE = 'AI_FUNCTION_EVALUATION'
  FROM SPECIFICATION $$
    query_text: AI_FUNCTIONS_EVAL_DEMO.EVAL.TRIAGE_TICKET(TICKET_TEXT)
    metrics:
      - name: llm_judge
        judge_model: claude-sonnet-4-6
    dataset:
      name: AI_FUNCTIONS_EVAL_DEMO.EVAL.TICKETS_HOLDOUT
      version: v1
      ground_truth: EXPECTED
    evaluation:
      num_eval_runs: 1
  $$""").collect()
    session.sql(f"EXECUTE EXPERIMENT {EVAL_EXP}").collect()
    print(f"Launched {EVAL_EXP}; runs asynchronously, re-run the next cell to check status")

In [ ]:
import json
rows = session.sql(f"SHOW RUNS IN EXPERIMENT AI_FUNCTIONS_EVAL_DEMO.EVAL.{EVAL_EXP}").collect()
for r in rows:
    m = json.loads(r["metadata"])
    print(r["name"], m["status"], "score =", m.get("metrics", {}).get("score", {}).get("value"))

### Why did it fail? The judge's per-row feedback

In [ ]:
session.sql("CREATE OR REPLACE TEMPORARY FILE FORMAT eval_detail_json_fmt TYPE = JSON STRIP_OUTER_ARRAY = TRUE").collect()
failures = session.sql(f"""
SELECT $1:row_id::INT AS row_id, $1:metric_score::FLOAT AS score, $1:metric_feedback::STRING AS judge_feedback
FROM 'snow://experiment/{EVAL_EXP}/versions/EVAL_1/eval_detail.json' (FILE_FORMAT => eval_detail_json_fmt)
WHERE $1:metric_score::FLOAT < 1 ORDER BY row_id""")
failures.limit(10)

## 4. Optimize: prompt tuning + model selection
The optimizer:
- seeds from the current function body
- reflects on the judge's feedback, proposes better prompts and post-processing, and keeps the Pareto-best candidates
- does this for **several models at once** (budget → premium), so cost vs quality becomes a measured choice instead of a guess
- scores the final candidates on a **held-out** set they never trained on

In [ ]:
if RUN_LIVE:
    session.sql(f"""
CREATE EXPERIMENT {OPT_EXP}
  TYPE = 'AI_FUNCTION_OPTIMIZATION'
  FROM SPECIFICATION $$
    function:
      function_name: AI_FUNCTIONS_EVAL_DEMO.EVAL.TRIAGE_TICKET
    metrics:
      - name: llm_judge
        judge_model: claude-sonnet-4-6
    dataset:
      name: AI_FUNCTIONS_EVAL_DEMO.EVAL.TICKETS_TRAIN
      version: v1
      column_mapping:
        argument_mapping:
          TICKET_TEXT: TICKET_TEXT
        ground_truth: EXPECTED
      holdout_data: AI_FUNCTIONS_EVAL_DEMO.EVAL.TICKETS_HOLDOUT
      holdout_version: v1
    optimization:
      models: [openai-gpt-5-nano, ministral-3-8b, claude-sonnet-4-6]
      reflection_model: claude-opus-4-7
      strategy: quality_first
      budget: light
  $$""").collect()
    session.sql(f"EXECUTE EXPERIMENT {OPT_EXP}").collect()
    print(f"Launched {OPT_EXP}")

In [ ]:
import pandas as pd
runs = session.sql(f"SHOW RUNS IN EXPERIMENT AI_FUNCTIONS_EVAL_DEMO.EVAL.{OPT_EXP}").collect()
run_df = pd.DataFrame([{"run": r["name"], **json.loads(r["metadata"])} for r in runs])
run_df["status"].value_counts()

### Every candidate: validation score vs relative cost
Each point is a candidate function body. `cost_compared_to_seed` is the cost relative to the original Sonnet function (1.0×).

In [ ]:
recs = []
for r in runs:
    if json.loads(r["metadata"])["status"] != "FINISHED":
        continue
    params  = {p["name"]: p["value"] for p in session.sql(f"SHOW RUN PARAMETERS IN EXPERIMENT AI_FUNCTIONS_EVAL_DEMO.EVAL.{OPT_EXP} RUN {r['name']}").collect()}
    metrics = {m["name"]: float(m["value"]) for m in session.sql(f"SHOW RUN METRICS IN EXPERIMENT AI_FUNCTIONS_EVAL_DEMO.EVAL.{OPT_EXP} RUN {r['name']}").collect()}
    recs.append({"run": r["name"], "model": params.get("model"), "run_type": params.get("run_type"), **metrics})
results = pd.DataFrame(recs).dropna(subset=["val_score"])
results.sort_values(["model", "val_score"], ascending=[True, False])

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 5))
for model, g in results.groupby("model"):
    ax.scatter(g["cost_compared_to_seed"], g["val_score"], label=model, s=60)
ax.set_xscale("log"); ax.set_xlabel("cost relative to seed (log)"); ax.set_ylabel("validation score")
ax.axhline(results.loc[results.run == "SEED", "val_score"].iloc[0], ls="--", c="gray", label="seed")
ax.set_title("Optimization candidates: quality vs cost"); ax.legend(); plt.show()

### What did the optimizer change? (best Sonnet candidate)

In [ ]:
best_run = results[results.model == "claude-sonnet-4-6"].sort_values("val_score", ascending=False).iloc[0]["run"]
best = {p["name"]: p["value"] for p in session.sql(f"SHOW RUN PARAMETERS IN EXPERIMENT AI_FUNCTIONS_EVAL_DEMO.EVAL.{OPT_EXP} RUN {best_run}").collect()}
print(best_run); print(best["function_impl"][:3500])

It rewrote the **whole function body**, not just the prompt:
- added `enum` constraints in `response_format`, so out-of-taxonomy values can't be emitted
- added explicit decision rules for ambiguous cases (defect vs how_to, neutral vs negative)
- *inferred* an escalation rule from the examples, and got it only approximately right

## 5. Validate on the held-out set: never trust the validation score
Deploy the best candidate per model **directly from the experiment** and score each on the 50 holdout tickets it never saw.

In [ ]:
CANDIDATES = {"TRIAGE_TICKET_OPT_SONNET": "ITER_21", "TRIAGE_TICKET_OPT_NANO": "ITER_31", "TRIAGE_TICKET_OPT_MINISTRAL": "ITER_9"}
if RUN_LIVE:
    for fn, run in CANDIDATES.items():
        session.sql(f"CREATE OR REPLACE AI FUNCTION AI_FUNCTIONS_EVAL_DEMO.EVAL.{fn}(TICKET_TEXT VARCHAR) RETURNS VARIANT "
                    f"FROM EXPERIMENT AI_FUNCTIONS_EVAL_DEMO.EVAL.{OPT_EXP} RUN {run}").collect()

**Deterministic per-field accuracy** complements the LLM judge: it shows *which* field fails, costs nothing, and is fully reproducible.

In [ ]:
FIELD_SQL = """
SELECT p.variant,
  ROUND(AVG(IFF(p.predicted:issue_type::STRING = h.expected:issue_type::STRING,1,0)),2) AS issue_type,
  ROUND(AVG(IFF(p.predicted:sentiment::STRING = h.expected:sentiment::STRING,1,0)),2) AS sentiment,
  ROUND(AVG(IFF(p.predicted:requested_action::STRING = h.expected:requested_action::STRING,1,0)),2) AS requested_action,
  ROUND(AVG(IFF(p.predicted:escalation_flag::BOOLEAN = h.expected:escalation_flag::BOOLEAN,1,0)),2) AS escalation_flag,
  ROUND(AVG(IFF(LOWER(p.predicted:product::STRING) = LOWER(h.expected:product::STRING)
     AND p.predicted:issue_type::STRING = h.expected:issue_type::STRING AND p.predicted:sentiment::STRING = h.expected:sentiment::STRING
     AND p.predicted:requested_action::STRING = h.expected:requested_action::STRING
     AND EQUAL_NULL(NULLIF(p.predicted:order_id::STRING,''), h.expected:order_id::STRING)
     AND p.predicted:escalation_flag::BOOLEAN = h.expected:escalation_flag::BOOLEAN,1,0)),2) AS all_fields
FROM AI_FUNCTIONS_EVAL_DEMO.EVAL.HOLDOUT_PREDICTIONS p JOIN AI_FUNCTIONS_EVAL_DEMO.EVAL.{labels} h USING (ticket_text)
GROUP BY 1 ORDER BY all_fields DESC"""
session.sql(FIELD_SQL.format(labels="HOLDOUT")).to_pandas()

Taxonomy fields jump from 0% to 80–98%, but **escalation_flag stalls at ~76%**, even after we added the explicit business rule.
When a field won't move no matter what you change in the prompt, **look at the labels.**

## 6. Review the disagreements: some "errors" are bad labels
A stronger model (with a human in the loop in real life) adjudicates only the rows where the model and the label disagree, using the written business rule.
Corrections become a **new immutable DATASET version** (`v2`), and every function is re-scored on it.

In [ ]:
%%sql -r label_review
SELECT r.original_label:escalation_flag::BOOLEAN AS old_esc, r.review:escalation_flag::BOOLEAN AS new_esc,
       r.review:changed_fields AS changed_fields, r.review:rationale::STRING AS rationale, LEFT(r.ticket_text, 160) AS ticket
FROM AI_FUNCTIONS_EVAL_DEMO.EVAL.LABEL_REVIEW r WHERE ARRAY_SIZE(r.review:changed_fields) > 0

In [ ]:
session.sql(FIELD_SQL.format(labels="HOLDOUT_V2")).to_pandas()

On the reviewed labels, the optimized Sonnet function **plus the explicit escalation rule (v2)** reaches **82% all-field exact match**
(baseline: 0%), with escalation at 92%.

> Caveat: only rows that the Sonnet variants disagreed on were adjudicated, which slightly favors those variants. In production, route every disagreement to human review.

In [ ]:
judge = {}
for exp in ["TRIAGE_BASELINE_EVAL_V2", "TRIAGE_OPT_NANO_EVAL_V2", "TRIAGE_OPT_MINISTRAL_EVAL_V2", "TRIAGE_OPT_SONNET_EVAL_V2", "TRIAGE_OPT_SONNET_V2_EVAL_V2"]:
    r = session.sql(f"SHOW RUNS IN EXPERIMENT AI_FUNCTIONS_EVAL_DEMO.EVAL.{exp}").collect()[0]
    judge[exp] = json.loads(r["metadata"])["metrics"]["score"]["value"]
pd.Series(judge, name="llm_judge score (holdout v2)")

## 7. Deploy to production
The final function is the experiment's best Sonnet body plus the explicit escalation rule (`03_optimized_function_v2.sql`).
For a pure experiment winner, deploy with `CREATE AI FUNCTION ... FROM EXPERIMENT <exp> RUN <run>`.

In [ ]:
session.sql("""CREATE OR REPLACE AI FUNCTION AI_FUNCTIONS_EVAL_DEMO.PROD.TRIAGE_TICKET(TICKET_TEXT VARCHAR) RETURNS VARIANT
AS $$ SELECT AI_FUNCTIONS_EVAL_DEMO.EVAL.TRIAGE_TICKET_OPT_SONNET_V2(TICKET_TEXT) $$""").collect() if RUN_LIVE else None

## 8. Production: incremental, tagged, versioned
`04_production_monitoring.sql` creates the pipeline. A **stream + task** triages new tickets every 5 minutes, and:
- writes the `function_version` with every result, so quality can be traced to a prompt/model
- sets a `QUERY_TAG` so cost is attributable in `CORTEX_AI_FUNCTIONS_USAGE_HISTORY`

In [ ]:
%%sql -r prod_results
SELECT processed_at, function_version, result:issue_type::STRING issue_type, result:sentiment::STRING sentiment,
       result:escalation_flag::BOOLEAN escalate, LEFT(ticket_text, 120) ticket
FROM AI_FUNCTIONS_EVAL_DEMO.PROD.TRIAGE_RESULTS ORDER BY processed_at DESC LIMIT 10

## 9. Monitor once in production
There's no ground truth in production, so we combine four signals:

| Signal | Question | Source |
|---|---|---|
| **Cost** | What does this workload cost per 1K tickets? Any spikes? | `CORTEX_AI_FUNCTIONS_USAGE_HISTORY` + `QUERY_TAG` |
| **Output health** | Are values inside the taxonomy? Nulls? Bad IDs? | deterministic SQL checks |
| **Drift** | Is the escalation or negative-sentiment rate moving away from the eval baseline? | prod vs golden-set distributions |
| **Sampled quality** | Does a reference-free judge agree on a daily sample? | `AI_COMPLETE` judge task |

In [ ]:
%%sql -r mon_cost
SELECT * FROM AI_FUNCTIONS_EVAL_DEMO.PROD.MON_COST_DAILY ORDER BY usage_day DESC

In [ ]:
%%sql -r mon_health
SELECT * FROM AI_FUNCTIONS_EVAL_DEMO.PROD.MON_OUTPUT_HEALTH ORDER BY hour DESC

In [ ]:
%%sql -r mon_drift
SELECT * FROM AI_FUNCTIONS_EVAL_DEMO.PROD.MON_DRIFT ORDER BY day DESC

In [ ]:
d = mon_drift.to_pandas() if hasattr(mon_drift, "to_pandas") else mon_drift
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, col, base, title in [(ax[0], "ESC_RATE", "BASELINE_ESC_RATE", "Escalation rate"),
                             (ax[1], "NEG_RATE", "BASELINE_NEG_RATE", "Negative sentiment rate")]:
    a.bar(["eval baseline", "production"], [d[base].iloc[0], d[col].iloc[0]], color=["#7f8c8d", "#29b5e8"])
    a.set_title(title); a.set_ylim(0, 1)
plt.show()

In [ ]:
%%sql -r judge_now
-- Run the daily quality-sample judge on demand
EXECUTE TASK AI_FUNCTIONS_EVAL_DEMO.PROD.JUDGE_SAMPLE_TASK

In [ ]:
%%sql -r mon_quality
SELECT q.ticket_id, q.verdict:all_correct::BOOLEAN all_correct, q.verdict:incorrect_fields incorrect_fields, q.verdict:reason::STRING reason
FROM AI_FUNCTIONS_EVAL_DEMO.PROD.QUALITY_SAMPLES q ORDER BY judged_at DESC LIMIT 20

### Closing the loop
- **Alert** `TRIAGE_QUALITY_ALERT` fires hourly on health, drift, or judge-accuracy thresholds.
- Judge-flagged rows are reviewed, labeled, and **added as a new DATASET version** → re-evaluate → re-optimize → redeploy.
- Gate every prompt or model change on the evaluation experiment, the same way you'd gate code on CI tests.

In [ ]:
%%sql -r alerts
SELECT * FROM AI_FUNCTIONS_EVAL_DEMO.PROD.ALERT_LOG ORDER BY fired_at DESC